In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.image import imread

# For better plot display
plt.style.use("ggplot")

from google.colab import drive

drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/self_driving_car_dataset_jungle"

csv_path = os.path.join(dataset_path, "driving_log.csv")

img_folder = os.path.join(dataset_path, "IMG")

# Define column names based on the typical format of Udacity self-driving car dataset logs
column_names = ['center', 'left', 'right', 'steering', 'throttle', 'reverse', 'speed']

# Load the CSV file, specifying that there is no header row and providing column names
df = pd.read_csv(csv_path, names=column_names)

print("Dataset Loaded Successfully!")
print("\nDataset Shape:")
print(df.shape)

df.head()

print("\nDataset Information:\n")

df.info()

print("\nMissing Values:\n")

print(df.isnull().sum())

steering_angles = df['steering']

print("\nSteering Statistics:\n")

print(steering_angles.describe())

plt.figure(figsize=(14,6))

sns.histplot(
    steering_angles,
    bins=50,
    kde=True
)

plt.title("Steering Angle Distribution", fontsize=16)

plt.xlabel("Steering Angle", fontsize=12)

plt.ylabel("Frequency", fontsize=12)

plt.show()

print("\nIMPORTANT OBSERVATION:")
print("Most steering angles are near 0.")
print("This means the dataset is heavily imbalanced toward straight driving.")

def get_image_path(image_path):
    # First, normalize Windows-style backslashes to forward slashes
    normalized_path = image_path.replace('\\', '/')
    # Then, extract only the filename using os.path.basename
    filename = os.path.basename(normalized_path)

    # Create correct full path by joining with the image folder
    full_path = os.path.join(img_folder, filename)

    return full_path

print("\nOriginal CSV Path:\n")

print(df['center'][0])

print("\nCleaned Image Path:\n")

print(get_image_path(df['center'][0]))

sample_image_path = get_image_path(df['center'][0])

image = cv2.imread(sample_image_path)

# Convert BGR to RGB
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("\nImage Shape:")

print(image.shape)

plt.figure(figsize=(12,6))

plt.imshow(image)

plt.title(
    f"Sample Driving Image\nSteering Angle: {df['steering'][0]:.4f}",
    fontsize=14
)

plt.axis("off")

plt.show()

fig, axes = plt.subplots(2, 3, figsize=(18,8))

for i, ax in enumerate(axes.flat):

    img_path = get_image_path(df['center'][i])

    img = cv2.imread(img_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax.imshow(img)

    ax.set_title(
        f"Steering: {df['steering'][i]:.4f}"
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

print("\nOriginal Image Dimensions:")

print(image.shape)

print("\nIMAGE OBSERVATIONS:")
print("- Upper region contains sky and mountains")
print("- Lower region contains vehicle hood")
print("- Middle region contains road and lane information")
print("- Not all regions are useful for steering prediction")

print("\nWHY PREPROCESSING IS IMPORTANT:")
print("- Removes unnecessary visual information")
print("- Helps CNN focus on road features")
print("- Reduces noise")
print("- Improves training efficiency")
print("- Improves model generalization")

print("\nPHASE 2 COMPLETED SUCCESSFULLY!")

print("\nWhat We Achieved:")
print("1. Loaded behavioral cloning dataset")
print("2. Explored steering angle distribution")
print("3. Identified steering imbalance problem")
print("4. Visualized driving images")
print("5. Built image path utilities")
print("6. Understood preprocessing requirements")

In [ ]:
def preprocess_image(image):
    cropped = image[60:135, :, :]
    resized = cv2.resize(cropped, (200, 66))
    yuv = cv2.cvtColor(resized, cv2.COLOR_RGB2YUV)
    blurred = cv2.GaussianBlur(yuv, (3,3), 0)
    normalized = blurred / 255.0
    return normalized

In [ ]:
# Load original image again

sample_image_path = get_image_path(df['center'][0])

original_image = cv2.imread(sample_image_path)

original_image = cv2.cvtColor(
    original_image,
    cv2.COLOR_BGR2RGB
)

# Preprocess image

processed_image = preprocess_image(original_image)

print("Original Shape:", original_image.shape)

print("Processed Shape:", processed_image.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))

# Original Image

axes[0].imshow(original_image)

axes[0].set_title("Original Image")

axes[0].axis("off")


# Processed Image

axes[1].imshow(processed_image)

axes[1].set_title("Processed Image")

axes[1].axis("off")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15,6))

for i, ax in enumerate(axes.flat):

    img_path = get_image_path(df['center'][i])

    img = cv2.imread(img_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    processed = preprocess_image(img)

    ax.imshow(processed)

    ax.set_title(
        f"Steering: {df['steering'][i]:.4f}"
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
import os

os.makedirs("/content/src", exist_ok=True)

In [ ]:
%%writefile /content/src/preprocessing.py

import cv2
import numpy as np


def preprocess_image(image):

    """
    Preprocess image for NVIDIA behavioral cloning model.
    """
    cropped = image[60:135, :, :]
    resized = cv2.resize(cropped, (200, 66))
    yuv = cv2.cvtColor(resized, cv2.COLOR_RGB2YUV)
    blurred = cv2.GaussianBlur(yuv, (3,3), 0)
    normalized = blurred / 255.0

    return normalized

In [ ]:
from src.preprocessing import preprocess_image

print("Preprocessing module imported successfully!")

In [ ]:
sample_path = get_image_path(df['center'][0])

image = cv2.imread(sample_path)

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

processed_image = preprocess_image(image)

print("Original Shape:", image.shape)

print("Processed Shape:", processed_image.shape)

In [ ]:
%%writefile /content/src/augmentation.py

import cv2
import numpy as np
import random


def random_flip(image, steering_angle):

    if random.random() < 0.5:

        image = cv2.flip(image, 1)

        steering_angle = -steering_angle

    return image, steering_angle


def random_brightness(image):

    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    brightness_scale = 1.0 + (np.random.rand() - 0.5) * 0.4

    hsv[:,:,2] = hsv[:,:,2] * brightness_scale

    hsv[:,:,2] = np.clip(hsv[:,:,2], 0, 255)

    image = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    return image


def random_shadow(image):

    """
    Add random shadow to simulate lighting variation.
    """

    height, width, _ = image.shape

    x1, y1 = width * np.random.rand(), 0
    x2, y2 = width * np.random.rand(), height

    xm, ym = np.mgrid[0:height, 0:width]

    mask = np.zeros_like(image[:,:,1])

    mask[
        (ym - y1) * (x2 - x1)
        - (y2 - y1) * (xm - x1) > 0
    ] = 1

    shadow_intensity = 0.5

    image_hls = cv2.cvtColor(image, cv2.COLOR_RGB2HLS)

    image_hls[:,:,1][mask == 1] = (
        image_hls[:,:,1][mask == 1] * shadow_intensity
    ).astype(np.uint8)

    image = cv2.cvtColor(image_hls, cv2.COLOR_HLS2RGB)

    return image


def random_translate(image, steering_angle,
                     range_x=100,
                     range_y=10):

    trans_x = range_x * (np.random.rand() - 0.5)

    trans_y = range_y * (np.random.rand() - 0.5)

    steering_angle += trans_x * 0.002

    translation_matrix = np.float32([
        [1, 0, trans_x],
        [0, 1, trans_y]
    ])

    height, width = image.shape[:2]

    image = cv2.warpAffine(
        image,
        translation_matrix,
        (width, height)
    )

    return image, steering_angle


def random_zoom(image):

    zoom_scale = 1 + (np.random.rand() * 0.3)

    height, width = image.shape[:2]

    new_height = int(height / zoom_scale)

    new_width = int(width / zoom_scale)

    y1 = np.random.randint(0, height - new_height)

    x1 = np.random.randint(0, width - new_width)

    cropped = image[
        y1:y1+new_height,
        x1:x1+new_width
    ]

    image = cv2.resize(cropped, (width, height))

    return image


def augment_image(image, steering_angle):

    image, steering_angle = random_flip(
        image,
        steering_angle
    )

    image, steering_angle = random_translate(
        image,
        steering_angle
    )

    image = random_brightness(image)

    image = random_shadow(image)

    image = random_zoom(image)

    return image, steering_angle

In [ ]:
from importlib import reload

import src.augmentation

reload(src.augmentation)

from src.augmentation import augment_image

In [ ]:
import sys

sys.path.append("/content")

from src.augmentation import augment_image

print("Augmentation module imported successfully!")

In [ ]:
sample_path = get_image_path(df['center'][0])

image = cv2.imread(sample_path)

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

steering = df['steering'][0]

augmented_image, augmented_steering = augment_image(
    image,
    steering
)

print("Original Steering:", steering)

print("Augmented Steering:", augmented_steering)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))

# Original Image

axes[0].imshow(image)

axes[0].set_title("Original Image")

axes[0].axis("off")


# Processed Image

axes[1].imshow(processed_image)

axes[1].set_title("Processed Image")

axes[1].axis("off")

plt.show()

In [ ]:
%%writefile /content/src/model.py

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Flatten,
    Dropout
)


def build_nvidia_model():

    model = Sequential()

    model.add(
        Conv2D(
            24,
            (5,5),
            strides=(2,2),
            activation='relu',
            input_shape=(66, 200, 3)
        )
    )

    model.add(
        Conv2D(
            36,
            (5,5),
            strides=(2,2),
            activation='relu'
        )
    )

    model.add(
        Conv2D(
            48,
            (5,5),
            strides=(2,2),
            activation='relu'
        )
    )

    model.add(
        Conv2D(
            64,
            (3,3),
            activation='relu'
        )
    )

    model.add(
        Conv2D(
            64,
            (3,3),
            activation='relu'
        )
    )

    model.add(Flatten())

    model.add(Dense(100, activation='relu'))

    model.add(Dropout(0.5))

    model.add(Dense(50, activation='relu'))

    model.add(Dense(10, activation='relu'))

    model.add(Dense(1))

    return model

In [ ]:
from src.model import build_nvidia_model

In [ ]:
model = build_nvidia_model()

print("NVIDIA CNN Model Built Successfully!")

In [ ]:
model.summary()

In [ ]:
from sklearn.model_selection import train_test_split

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

In [ ]:
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(train_df))

print("Validation Samples:", len(valid_df))

In [ ]:
def batch_generator(dataframe, batch_size, training_flag):

    images = np.empty(
        [batch_size, 66, 200, 3]
    )

    steerings = np.empty(batch_size)

    while True:

        batch_indices = np.random.randint(
            0,
            len(dataframe),
            batch_size
        )

        for i, idx in enumerate(batch_indices):

            row = dataframe.iloc[idx]

            img_path = get_image_path(row['center'])

            image = cv2.imread(img_path)

            image = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

            steering = row['steering']
            if training_flag:

                image, steering = augment_image(
                    image,
                    steering
                )
            image = preprocess_image(image)

            images[i] = image

            steerings[i] = steering

        yield images, steerings

In [ ]:
model = build_nvidia_model()

print("Model Ready!")

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='mse'
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "behavioral_cloning_model.keras",
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
history = model.fit(
    batch_generator(
        train_df,
        batch_size=32,
        training_flag=True
    ),

    steps_per_epoch=30,

    epochs=10,

    validation_data=batch_generator(
        valid_df,
        batch_size=32,
        training_flag=False
    ),

    validation_steps=10,

    callbacks=[
        early_stop,
        checkpoint
    ],

    verbose=1
)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model(
    "behavioral_cloning_model.keras"
)

print("Best model loaded successfully!")

In [ ]:
sample_indices = [10, 50, 100, 200, 300]

fig, axes = plt.subplots(
    len(sample_indices),
    2,
    figsize=(12,20)
)

for i, idx in enumerate(sample_indices):

    row = df.iloc[idx]

    img_path = get_image_path(row['center'])

    image = cv2.imread(img_path)

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    actual_steering = row['steering']
    processed = preprocess_image(image)
    prediction = best_model.predict(
        np.expand_dims(processed, axis=0),
        verbose=0
    )

    predicted_steering = prediction[0][0]
    axes[i,0].imshow(image)

    axes[i,0].set_title(
        f"Actual: {actual_steering:.4f}"
    )

    axes[i,0].axis("off")
    axes[i,1].imshow(processed)

    axes[i,1].set_title(
        f"Predicted: {predicted_steering:.4f}"
    )

    axes[i,1].axis("off")

plt.tight_layout()

plt.show()

In [ ]:
actual_values = []

predicted_values = []

sample_size = 200


for i in range(sample_size):

    row = df.iloc[i]

    img_path = get_image_path(row['center'])

    image = cv2.imread(img_path)

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    processed = preprocess_image(image)

    prediction = best_model.predict(
        np.expand_dims(processed, axis=0),
        verbose=0
    )

    predicted_steering = prediction[0][0]

    actual_values.append(row['steering'])

    predicted_values.append(predicted_steering)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    actual_values,
    label='Actual Steering',
    linewidth=2
)

plt.plot(
    predicted_values,
    label='Predicted Steering',
    linewidth=2
)

plt.title("Actual vs Predicted Steering")

plt.xlabel("Sample Index")

plt.ylabel("Steering Angle")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
print("MODEL EVALUATION SUMMARY\n")

print("Strengths:")
print("- Learns steering behavior directly from images")
print("- CNN extracts spatial road features")
print("- Augmentation improves generalization")
print("- Preprocessing improves feature focus")

print("\nLimitations:")
print("- Small dataset size")
print("- Steering imbalance")
print("- Behavioral cloning may struggle with unseen roads")
print("- No temporal memory (single-frame prediction)")

In [ ]:
def predict_steering(model, image_path):

    """
    Predict steering angle from image.
    """
    image = cv2.imread(image_path)

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
    processed = preprocess_image(image)
    processed = np.expand_dims(
        processed,
        axis=0
    )
    prediction = model.predict(
        processed,
        verbose=0
    )

    steering_angle = prediction[0][0]

    return image, steering_angle

In [ ]:
test_index = 500

row = df.iloc[test_index]

test_image_path = get_image_path(
    row['center']
)

original_image, predicted_steering = predict_steering(
    best_model,
    test_image_path
)

actual_steering = row['steering']

print("Actual Steering:", actual_steering)

print("Predicted Steering:", predicted_steering)

In [ ]:
plt.figure(figsize=(12,6))

plt.imshow(original_image)

plt.title(
    f"Actual: {actual_steering:.4f} | Predicted: {predicted_steering:.4f}",
    fontsize=14
)

plt.axis("off")

plt.show()

In [ ]:
random_indices = np.random.randint(
    0,
    len(df),
    5
)

fig, axes = plt.subplots(5, 1, figsize=(12,20))

for i, idx in enumerate(random_indices):

    row = df.iloc[idx]

    img_path = get_image_path(
        row['center']
    )

    image, prediction = predict_steering(
        best_model,
        img_path
    )

    actual = row['steering']

    axes[i].imshow(image)

    axes[i].set_title(
        f"Actual: {actual:.4f} | Predicted: {prediction:.4f}"
    )

    axes[i].axis("off")

plt.tight_layout()

plt.show()

In [ ]:
from google.colab import files

files.download("behavioral_cloning_model.keras")

In [ ]:
best_model.save_weights(
    "behavioral_cloning_weights.weights.h5"
)

In [ ]:
from google.colab import files

files.download(
    "behavioral_cloning_weights.weights.h5"
)